In [48]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# ------1. Initialize Embeddings ----

# We use a lightweight model suitable for CPU
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
# Creates the embedding model using the specified Sentence Transformer.
# model_name selects which pre-trained embedding model to use.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 506.53it/s]


In [49]:
content = """
LangChain is a Framework for developing applications powered by language models.
It enables applications that are context_aware and reason across problems.
Key components of LangChain include Models, Prompts, Chains and Memory.
Chains Allow you to combine multiple compinents together.
RAG (Retrieval-Augmented Generation) is a popular technique used in LangChain."""

In [50]:
embedding = embedding_function.embed_query(content)


embedding

[-0.04221038892865181,
 0.017317643389105797,
 0.01618487387895584,
 -0.04219436272978783,
 -0.04757053032517433,
 0.017276525497436523,
 -0.0004874477453995496,
 0.04401653632521629,
 0.07288435846567154,
 -0.05958278477191925,
 0.028378324583172798,
 0.00027472665533423424,
 0.05476345121860504,
 0.00394807942211628,
 0.014382054097950459,
 -0.003423909656703472,
 -0.026249047368764877,
 0.11966319382190704,
 0.010423804633319378,
 -0.10416429489850998,
 -0.020063072443008423,
 0.007850569672882557,
 -0.016696378588676453,
 0.025663726031780243,
 0.006298702675849199,
 -0.04688847064971924,
 0.006353402975946665,
 -0.03609039634466171,
 0.08026612550020218,
 0.011479820124804974,
 0.035490863025188446,
 0.11452947556972504,
 -0.06773541122674942,
 0.006397343706339598,
 -0.14679698646068573,
 0.13120922446250916,
 -0.023908760398626328,
 0.012161919847130775,
 -0.00691381236538291,
 -0.05000188574194908,
 -0.07754908502101898,
 0.045265767723321915,
 0.05008988827466965,
 -0.06542734

In [51]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---- 1. Create a Sample Documents ------
content = """
LangChain is a Framework for developing applications powered by language models.
It enables applications that are context_aware and reason across problems.
Key components of LangChain include Models, Prompts, Chains and Memory.
Chains Allow you to combine multiple compinents together.
RAG (Retrieval-Augmented Generation) is a popular technique used in LangChain."""

with open("langchian_intro.txt", 'w', encoding="utf-8") as f:
      f.write(content)


# ----- 2. Load the Document -----

loader = TextLoader("langchian_intro.txt", encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} documents(s).")

# ---3. Split Documents (Chunking) ----
# Splits the document into smaller pieces for easier processing.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,    # Sets the maximum target size of each chunk.
    chunk_overlap=20,  # Keeps some text from the previous chunk for context continuity.
    length_function=len,  # Uses Python's len() function to measure text length.
    separators=["\n\n", "\n", " ", ""]  # Defines where the splitter should try to split the text.
)

splits = text_splitter.split_documents(documents)

print(f"Split into {len(splits)} chunks.")

for i, split in enumerate(splits):
    print(f"Chunk {i}:{split.page_content}")

Loaded 1 documents(s).
Split into 5 chunks.
Chunk 0:LangChain is a Framework for developing applications powered by language models.
Chunk 1:It enables applications that are context_aware and reason across problems.
Chunk 2:Key components of LangChain include Models, Prompts, Chains and Memory.
Chunk 3:Chains Allow you to combine multiple compinents together.
Chunk 4:RAG (Retrieval-Augmented Generation) is a popular technique used in LangChain.


# Embeddings & Vector Database (FAISS)

## Objective:

Based in The "Embeddings" and "Vector Database" slides. Convert text chunks into vectors and store them In FALSS for fast similarity search.

In [52]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
# ---- 1. Initialize Embaddings ------

# We Use lightweight model suitable for CPU

embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# ---- 2. Create Embaddings and Index in FAISS ------

# Slide: "FAISS allow fast similarity search of chunks"

print("Creating embeddings and Indexing in FAISS....")

db = FAISS.from_documents(
    documents=splits,
    embedding=embedding_function
)

print (" Vector Database Created Successfuly!")

print(f" Stored {len(splits)} vectors.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 860.78it/s]


Creating embeddings and Indexing in FAISS....
 Vector Database Created Successfuly!
 Stored 5 vectors.


# Download The LLM & Prepare prompt for RAG:

In [53]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# ------Configuration -------

model_id = "Qwen/Qwen2-0.5B-Instruct"

# ---- 1. Load Model & Tokenizer ---

print(f"Loading model: {model_id}....")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",    # Use float16 if GPU avalible
    device_map="auto"      # Automatically Use GPU if present, else CPU
)

#  ---- 2. Create HF Pipeline ----

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer = tokenizer,
    max_new_tokens=128,     #Limit length for speed
    temperature=0.1,        # Lower temerature for less creativity / hallucination
    do_sample=True          # Enable sampling if needed, but low temp makes it daterminstic
)

# --- 3. LangChain LLM Wrapper ----
raw_llm = HuggingFacePipeline(pipeline=pipe)

 # --- 4. Define the Qwen chat Formatter -----
 # This formats inputs as chat messages using Qwen's Template
def format_for_qwen(input_dict):
    # Input_dict has "context" and "Question" (Renamed for 'input')
    messages = [
        {"role": "system", "content": """
You are a helpful AI assistant that answers questions strictly based on the provided context.
Do not use any External Knowledge or make up information.
If the Answer is not in the context, respond exactly with: "I don't know based on the provided context."
Always Start Your response with "Answer:" followed by the answer or the "I don't know" Statement.
"""},
        {"role":"user", "content":f"""
Context:
{input_dict["context"]}
Question:
{input_dict["question"]}
"""}
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False, 
        add_generation_prompt=True
    )
    return formatted_prompt

# --- 5. Generation Function --- 

def generate_with_qwen(formatted_prompt):
        # Use .invokw() instead of direct call
    response = raw_llm.invoke(formatted_prompt)
        # Extract generated text (after prompt)
    generated = response.split(formatted_prompt)[-1].strip()
        # Further clean: if it strat with "Answer", extract after it
    if generated.startswith("Answer"):
        generated = generated.split("Answer", 1)[-1].strip()
    return generated

# --- 6. Create LLM Chain with Formatting ----

llm_with_format = (
    RunnableLambda(format_for_qwen)
    | RunnableLambda(generate_with_qwen)
    | StrOutputParser()
)
print("LLM Setup Complete")

Loading model: Qwen/Qwen2-0.5B-Instruct....


Loading weights: 100%|██████████| 290/290 [00:07<00:00, 38.51it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


LLM Setup Complete


# Building the RAG Pipeline

## Objective:

Based on the "Building a RAG Pipeline" and "Step 4: Build Retriever + RetrievalQA Chain" slides, combine the retriever and the generator (Qwen) to answer a user question using the retrieved context.


In [54]:
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# ---- 1. Define the Retriever ----

retriever = db.as_retriever(search_kwargs={"k":2})

In [55]:
# --- 2. Create the RAG Chain ----
# Retrival -> Format context & Question -> LLM -> Output

rag_chain = (
    {"context":retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
     "question":RunnablePassthrough()}
     | llm_with_format
)

In [56]:
# --- 3. Query Example ----
# Retrival -> Format context & Question -> LLM -> Output

question = "What dose RAG Stand for?"
print(f"Question:{question}\n")

# Invoke the Chain
response = rag_chain.invoke(question)
print("RAG Answer:")
print(response)

print("\n ---- Context Used ----")
# Corrected: Use .invoke() on the retriver to get documents
contaxt_docs = retriever.invoke(question)
for doc in contaxt_docs:
    print(doc.page_content)

Question:What dose RAG Stand for?



[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG Answer:
RAG stands for Retrieval-Augmented Generation.

 ---- Context Used ----
RAG (Retrieval-Augmented Generation) is a popular technique used in LangChain.
Chains Allow you to combine multiple compinents together.
